## Example 1: Error Classes and Exceptions

In [ ]:
# Example 1: Error Classes and Exceptions
import snowflake.connector

print("=== Error Classes and Exceptions ===")

# Check available error classes
error_classes = []
for attr in dir(snowflake.connector):
    obj = getattr(snowflake.connector, attr)
    if isinstance(obj, type) and 'Error' in attr:
        error_classes.append(attr)

print(f"Error classes found: {len(error_classes)}")
for err in sorted(error_classes)[:10]:
    print(f"  - {err}")

# Demonstrate error handling structure
try:
    # This would fail without valid credentials
    raise snowflake.connector.errors.DatabaseError("Demo error")
except snowflake.connector.errors.DatabaseError as e:
    # The connector's DB-API error keeps the message and carries its own errno field (-1 when none is given).
    assert e.msg == "Demo error" and str(e) == "Demo error", (e.msg, str(e))
    assert isinstance(e, snowflake.connector.errors.Error)
    print(f"\nCaught {type(e).__name__} as expected")
    print(f"Message: {e.msg} (errno {e.errno})")
else:
    raise AssertionError("expected snowflake.connector.errors.DatabaseError")

print("Example 1 completed!")

## Example 2: The pandas / Arrow result path, with no server

In [ ]:
import decimal, io
import pandas as pd
import pyarrow as pa
from snowflake.connector.arrow_context import ArrowConverterContext
from snowflake.connector.nanoarrow_arrow_iterator import PyArrowTableIterator

cols = {"n": (pa.array([-12345, None], pa.int64()), {"logicalType": "FIXED", "scale": "2", "precision": "38"}),
        "r": (pa.array([float("nan"), None], pa.float64()), {"logicalType": "REAL"}),
        "s": (pa.array(["\u00fc", None], pa.string()), {"logicalType": "TEXT"})}
schema = pa.schema([pa.field(name, arr.type, True, meta) for name, (arr, meta) in cols.items()])
sink = io.BytesIO()
with pa.ipc.new_stream(sink, schema) as writer:
    writer.write_batch(pa.record_batch([arr for arr, _ in cols.values()], schema=schema))
# The cursor passes a context built from the session parameters, which always carry TIMEZONE.
table = next(PyArrowTableIterator(None, sink.getvalue(), ArrowConverterContext({"TIMEZONE": "UTC"}), False, False, True, True))
assert str(table.schema.field("n").type) == "decimal128(38, 2)", table.schema
df = table.to_pandas()
assert df["n"].tolist() == [decimal.Decimal("-123.45"), None], df["n"].tolist()
assert pd.api.types.is_string_dtype(df["s"]) and df["s"][0] == "\u00fc" and df["s"].isna().tolist() == [False, True], df["s"]
# NaN versus NULL: the default float64 frame folds both into NaN; an Arrow-backed frame keeps them apart.
assert df["r"].isna().tolist() == [True, True], df["r"].tolist()
assert table.to_pandas(types_mapper=pd.ArrowDtype)["r"].isna().tolist() == [False, True]
print(f"pyarrow {pa.__version__}, pandas {pd.__version__}: {dict(df.dtypes.astype(str))}")
print("Example 2 completed!")

## Example 3: The compiled Arrow row decoder, run on real bytes

In [ ]:
import datetime, decimal, io
import pyarrow as pa
from snowflake.connector.arrow_context import ArrowConverterContext
from snowflake.connector.errors import InterfaceError
from snowflake.connector.nanoarrow_arrow_iterator import PyArrowRowIterator

def decode(**columns):  # name=(int64 values, Snowflake column metadata)
    schema = pa.schema([pa.field(name, pa.int64(), True, meta) for name, (_, meta) in columns.items()])
    sink = io.BytesIO()
    with pa.ipc.new_stream(sink, schema) as writer:
        writer.write_batch(pa.record_batch([pa.array(v, pa.int64()) for v, _ in columns.values()], schema=schema))
    return list(PyArrowRowIterator(None, sink.getvalue(), ArrowConverterContext(), False, False, False, True))

rows = decode(t=([45296_123456789, None], {"logicalType": "TIME", "scale": "9"}),  # 12:34:56.123456789
              ts=([-1500, 1_700_000_000_123], {"logicalType": "TIMESTAMP_NTZ", "scale": "3"}),  # 1.5 s before the epoch
              n=([-12345, 99], {"logicalType": "FIXED", "scale": "2", "precision": "38"}),
              i=([2**62, -7], {"logicalType": "FIXED", "scale": "0", "precision": "19"}))
assert rows == [(datetime.time(12, 34, 56, 123456), datetime.datetime(1969, 12, 31, 23, 59, 58, 500000), decimal.Decimal("-123.45"), 2**62),
                (None, datetime.datetime(2023, 11, 14, 22, 13, 20, 123000), decimal.Decimal("0.99"), -7)], rows
try:  # a scale outside 0-9 must be refused, never turned into a wrong time
    decode(t=([0], {"logicalType": "TIME", "scale": "10"}))
except InterfaceError as e:
    assert "invalid scale value 10" in str(e), e
else:
    raise AssertionError("TIME scale 10 was accepted")
print(f"Decoded {len(rows)} rows exactly; an out-of-range scale is refused.")
print("Example 3 completed!")